In [ ]:
import torch
import torch.nn as nn
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import numpy as np

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define how the images should be processed (Matches your Keras shape)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), # Converts image to PyTorch Tensor and scales to 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

# 2. Point to your train and validation folders
# (Replace 'path/to/train' with your actual directory paths)
train_dataset = datasets.ImageFolder(root='../../../tf-venv/NIDS/CICD/new_Images/train/', transform=transform)
val_dataset = datasets.ImageFolder(root='../../../tf-venv/NIDS/CICD/new_Images/val', transform=transform)

# 3. Create the DataLoaders (These replace your Keras generators)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import timm

# Set up GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLASSES = 6

# Swapped 'levit_256' for a MobileViT model (e.g., 'mobilevit_s' for Small)
model = timm.create_model('mobilevit_s', pretrained=True, num_classes=NUM_CLASSES)

# Move the model to your GPU
model = model.to(device)

In [ ]:
from tqdm.auto import tqdm
# Define your loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- EARLY STOPPING SETUP ---
EPOCHS = 10 # You can set this higher now; early stopping will catch it
patience = 2 # How many epochs to wait for improvement before stopping
patience_counter = 0
best_val_loss = float('inf')
best_model_path = 'mobilevit_best_model.pth'

print("\n--- Starting PyTorch Training ---")
for epoch in range(EPOCHS):
    # ==========================
    # TRAINING PHASE
    # ==========================
    model.train() # Tell the model it is in training mode
    running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=True)
    
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()           # Clear old gradients
        outputs = model(images)         # Forward pass (predict)
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                 # Backward pass
        optimizer.step()                # Update weights
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    # ==========================
    # VALIDATION PHASE
    # ==========================
    model.eval() # Switch to evaluation mode
    val_running_loss = 0.0
    
    with torch.no_grad():
        # We don't necessarily need tqdm for validation inside the loop, to keep output clean
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()
            
    epoch_val_loss = val_running_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}")

    # ==========================
    # EARLY STOPPING LOGIC
    # ==========================
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        # Save the model weights since this is the best we've seen so far
        torch.save(model.state_dict(), best_model_path)
        print("  --> Validation loss improved! Saved best model.")
    else:
        patience_counter += 1
        print(f"  --> No improvement. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print("\n[!] Early stopping triggered! Training halted.")
        break

# ==========================
# FINAL EVALUATION
# ==========================
print("\n--- Final Evaluation ---")
# Load the best weights before doing the final classification report
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Assuming train_dataset is accessible here to get classes
class_names = train_dataset.classes 
print(class_names)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
model.eval()
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():  # No gradients needed
    for images, labels in tqdm(val_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        class_names = train_dataset.classes
print(class_names)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
best_model_path='Levit256.pth'
torch.save(model.state_dict(), best_model_path)

In [ ]:
import timm

# Set up GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLASSES = 6

# timm automatically drops the 1000-class ImageNet head and adds a fresh 6-class head!
model = timm.create_model('efficientformer_l1', pretrained=True, num_classes=NUM_CLASSES)
# Move the model to your GPU
model = model.to(device)

In [ ]:
from tqdm.auto import tqdm
# Define your loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 1

print("\n--- Starting PyTorch Training ---")
for epoch in range(EPOCHS):
    model.train() # Tell the model it is in training mode
    running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=True)
    
    # Iterate through your training data using the progress bar
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device) # Move data to GPU
        
        optimizer.zero_grad()           # Clear old gradients
        outputs = model(images)         # Forward pass (predict)
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                 # Backward pass
        optimizer.step()                # Update weights
        
        # Track accuracy and loss
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")
model.eval()
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():  # No gradients needed
    for images, labels in tqdm(val_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        class_names = train_dataset.classes
print(class_names)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
import torch
import torch.nn as nn
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report
import numpy as np

# Set up GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLASSES = 6

# timm automatically drops the 1000-class ImageNet head and adds a fresh 6-class head!
model = timm.create_model('efficientvit_b1', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

# Define your loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- EARLY STOPPING SETUP ---
EPOCHS = 5 # You can set this higher now; early stopping will catch it
patience = 2 # How many epochs to wait for improvement before stopping
patience_counter = 0
best_val_loss = float('inf')
best_model_path = 'best_swin_model.pth'

print("\n--- Starting PyTorch Training ---")
for epoch in range(EPOCHS):
    # ==========================
    # TRAINING PHASE
    # ==========================
    model.train() # Tell the model it is in training mode
    running_loss = 0.0
    correct = 0
    total = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=True)
    
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()           # Clear old gradients
        outputs = model(images)         # Forward pass (predict)
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                 # Backward pass
        optimizer.step()                # Update weights
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    # ==========================
    # VALIDATION PHASE
    # ==========================
    model.eval() # Switch to evaluation mode
    val_running_loss = 0.0
    
    with torch.no_grad():
        # We don't necessarily need tqdm for validation inside the loop, to keep output clean
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()
            
    epoch_val_loss = val_running_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}")

    # ==========================
    # EARLY STOPPING LOGIC
    # ==========================
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        # Save the model weights since this is the best we've seen so far
        torch.save(model.state_dict(), best_model_path)
        print("  --> Validation loss improved! Saved best model.")
    else:
        patience_counter += 1
        print(f"  --> No improvement. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print("\n[!] Early stopping triggered! Training halted.")
        break

# ==========================
# FINAL EVALUATION
# ==========================
print("\n--- Final Evaluation ---")
# Load the best weights before doing the final classification report
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Assuming train_dataset is accessible here to get classes
class_names = train_dataset.classes 
print(class_names)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Recreate the model architecture
NUM_CLASSES = 6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = timm.create_model('levit_256', pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load('Levit256.pth'))
model = model.to(device)
model.eval()

#test_paths = test_dataset.samples
import pandas as pd
import os
import torch.nn.functional as F  # IMPORTANT
prefix="trans"
def generate_predictions(model, loader, dataset, output_csv):
    model.eval()
    all_rows = []
    
    all_preds = []
    all_labels = []
    all_paths = []
    all_confidences = []   # NEW

    with torch.no_grad():
        idx = 0

        for images, labels in tqdm(loader):
            images = images.to(device)

            outputs = model(images)

            probs = F.softmax(outputs, dim=1)

            confs, preds = torch.max(probs, 1)

            probs_np = probs.cpu().numpy()

            batch_size = images.size(0)

            for i in range(batch_size):

                path, _ = dataset.samples[idx]
                filename = os.path.basename(path)

                row = {
                    "image_name": filename,
                    "true_label": dataset.classes[labels[i].item()],
                    f"{prefix}_pred": dataset.classes[preds[i].item()],
                    f"{prefix}_confidence": confs[i].item()
                }

                # Save ALL probabilities
                for c, class_name in enumerate(dataset.classes):
                    row[f"{prefix}_{class_name}"] = probs_np[i][c]

                all_rows.append(row)

                idx += 1

    df = pd.DataFrame(all_rows)

    df.to_csv(output_csv, index=False)

    print(f"Saved: {output_csv}")

In [ ]:
generate_predictions(model, train_loader, train_dataset, "levit_train_predictions.csv")


In [ ]:
generate_predictions(model, val_loader, val_dataset, "levit_val_predictions.csv")

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
# 1. Define how the images should be processed (Matches your Keras shape)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), # Converts image to PyTorch Tensor and scales to 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])
test_dataset = datasets.ImageFolder(root='../../../tf-venv/NIDS/CICD/Images/test', transform=transform)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
generate_predictions(model, test_loader, test_dataset, "levit_test_predictions.csv")

In [ ]:
from collections import Counter

labels = [label for _, label in train_dataset.samples]
class_counts = Counter(labels)

for class_idx, count in class_counts.items():
    print(f"{train_dataset.classes[class_idx]}: {count}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
df = pd.read_csv("tran_prob/val_predictions.csv")

cm = confusion_matrix(df["true_label"], df["trans_pred"], labels=val_dataset.classes)

In [ ]:
import matplotlib.pyplot as plt

class_names = val_dataset.classes

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

# Add numbers inside cells
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
plt.imshow(cm_normalized)
plt.title("Normalized Confusion Matrix")
plt.colorbar()

plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, f"{cm_normalized[i, j]:.2f}", ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()
plt.show()

In [ ]:
print("\n--- New Classification Report for Transformer (after modifications) ---")
print(classification_report(df["true_label"], df["trans_pred"], target_names=class_names))

In [ ]:
print("\n--- New Classification Report for Transformer (after modifications) ---")
print(classification_report(df["true_label"], df["trans_pred"], target_names=class_names))